# Scrap and chunk knowledge

Crawls Hate-Speech and Symbol Terminology references online and produces structured text chunks for `index.ipynb`.

- *Output*: corpus/chunks/chunks_knowledge.csv (and others less important)
- *Schema*: chunk_id | text  (same as chunks_example.csv)
- *Convention*: all entries are labeled [hate] since every symbol in the database is hate-associated by definition.

## 1. Imports

Standard data libraries (pandas, numpy, matplotlib), the Firecrawl client for web scraping, and HuggingFace transformers for tokenization.

In [ ]:
import os
import re
import pandas as pd
from urllib.parse import urljoin
from firecrawl import Firecrawl
from pathlib import Path

import numpy as np
from collections import Counter
import matplotlib.pyplot as plt
from transformers import AutoTokenizer

DOCS = Path("../../corpus/knowledge_documents")
SOURCES = DOCS / "sources"
CHUNKS = Path("../../corpus/chunks")

## 2. Firecrawl API Setup

Firecrawl requires a personal API key. Set it in the cell below before running. The key is read from the environment so it is never hardcoded in reproducible cells.

In [ ]:
# PRIVATE API key 
# Replace with your actual API key before running !
import os
os.environ["FIRECRAWL_API_KEY"] = "fc-enter_personal_api_key_here"

In [ ]:
FIRECRAWL_API_KEY = os.environ.get("FIRECRAWL_API_KEY")

if not FIRECRAWL_API_KEY:
    raise ValueError(
        "Missing FIRECRAWL_API_KEY.\n\n"
        "Set it first with:\n\n"
        "import os\n"
        "os.environ['FIRECRAWL_API_KEY'] = 'fc-your-real-key'"
    )

firecrawl = Firecrawl(api_key=FIRECRAWL_API_KEY)
print("Firecrawl client initialized.")

## 3. Source URL Discovery

This section crawls the ADL Hate Symbols Database and the SPLC Extremist Files listing pages to collect individual symbol/group page URLs. Each source produces a TXT file under `corpus/knowledge_documents/sources/` listing one `website,slug` entry per line.

In [ ]:
# Helper functions

def get_field(obj, key, default=None):
    if isinstance(obj, dict):
        return obj.get(key, default)
    return getattr(obj, key, default)


def preview_text(text, n=300):
    text = re.sub(r"\s+", " ", str(text)).strip()
    return text[:n] + ("..." if len(text) > n else "")


def url_to_website_slug(url):
    url = url.rstrip("/")
    website, slug = url.rsplit("/", 1)
    return website, slug


def extract_links_from_markdown(markdown, config):
    links = re.findall(r"\[([^\]]+)\]\(([^)]+)\)", markdown)

    urls = []

    domain = config["domain"]
    detail_path = config["detail_path"]
    skip_patterns = config.get("skip_patterns", [])

    for link_text, href in links:
        href = href.strip()

        if detail_path not in href:
            continue

        full_url = urljoin(domain, href)
        full_url = full_url.split("#")[0].split("?")[0].rstrip("/")

        should_skip = False

        for pat in skip_patterns:
            if re.search(pat, full_url):
                should_skip = True
                break

        if should_skip:
            continue

        urls.append(full_url)

    return urls


def build_page_url(config, page_num):
    pagination_type = config["pagination_type"]
    base_url = config["base_url"]

    if pagination_type == "query_page_zero_based":
        if page_num == 0:
            return base_url
        return f"{base_url}?page={page_num}"

    elif pagination_type == "path_page_one_based":
        if page_num == 1:
            return base_url.rstrip("/") + "/"
        return f"{base_url.rstrip('/')}/page/{page_num}/"

    else:
        raise ValueError(f"Unknown pagination_type: {pagination_type}")


SOURCE_CONFIGS = {
    "adl": {
        "source_name": "adl",
        "display_name": "ADL Hate Symbols Database",
        "base_url": "https://www.adl.org/resources/hate-symbols/search",
        "pagination_type": "query_page_zero_based",
        "start_page": 0,
        "end_page": 10,
        "domain": "https://www.adl.org",
        "detail_path": "/resources/hate-symbol/",
        "output_txt": SOURCES / "adl.txt",
        "review_csv": SOURCES / "adl_symbol_sources_review.csv",
        "log_csv": SOURCES / "adl_symbol_source_page_logs.csv",
        "skip_patterns": [
            r"/resources/hate-symbols/search",
        ],
    },

    "splc": {
        "source_name": "splc",
        "display_name": "SPLC Extremist Files",
        "base_url": "https://www.splcenter.org/resources/extremist-files",
        "pagination_type": "path_page_one_based",
        "start_page": 1,
        "end_page": 27,
        "domain": "https://www.splcenter.org",
        "detail_path": "/resources/extremist-files/",
        "output_txt": SOURCES / "splc.txt",
        "review_csv": SOURCES / "splc_sources_review.csv",
        "log_csv": SOURCES / "splc_source_page_logs.csv",
        "skip_patterns": [
            r"^https://www\.splcenter\.org/resources/extremist-files/?$",
            r"/resources/extremist-files/page/\d+$",
        ],
    },
}


def generate_source_txt(config):
    source_name = config["source_name"]
    display_name = config["display_name"]

    all_urls = []
    page_logs = []

    start_page = config["start_page"]
    end_page = config["end_page"]

    for page_num in range(start_page, end_page + 1):
        page_url = build_page_url(config, page_num)

        try:
            page = firecrawl.scrape(
                page_url,
                formats=["markdown"],
                only_main_content=True,
            )

            markdown = get_field(page, "markdown", "") or ""
            urls = extract_links_from_markdown(markdown=markdown, config=config)

            all_urls.extend(urls)

            page_logs.append({
                "source": source_name,
                "page_num": page_num,
                "page_url": page_url,
                "markdown_length": len(markdown),
                "links_found": len(urls),
                "status": "ok",
            })

        except Exception as e:
            print(f"  FAILED page {page_num}: {type(e).__name__}: {e}")

            page_logs.append({
                "source": source_name,
                "page_num": page_num,
                "page_url": page_url,
                "markdown_length": 0,
                "links_found": 0,
                "status": f"failed: {type(e).__name__}: {e}",
            })

    all_urls = sorted(set(all_urls))

    if len(all_urls) == 0:
        raise ValueError(f"No URLs found for {source_name}.")

    rows = []
    for url in all_urls:
        website, slug = url_to_website_slug(url)
        rows.append({"website": website, "slug": slug, "url": url, "source": source_name})

    df_sources = pd.DataFrame(rows).drop_duplicates(subset=["website", "slug"]).reset_index(drop=True)

    os.makedirs(SOURCES, exist_ok=True)

    txt_path = config["output_txt"]
    csv_path = config["review_csv"]
    log_path = config["log_csv"]

    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(f"# {display_name}\n")
        f.write("# Format: website,slug\n")
        f.write(f"# Generated from {config['base_url']}\n")
        f.write("\n")
        for _, row in df_sources.iterrows():
            f.write(f"{row['website']},{row['slug']}\n")

    df_sources.to_csv(csv_path, index=False)
    pd.DataFrame(page_logs).to_csv(log_path, index=False)

    print(f"Saved {len(df_sources):,} URLs to {txt_path}")

    return df_sources, pd.DataFrame(page_logs)


# Run all configured sources
all_source_dfs = []
all_log_dfs = []

for source_key, config in SOURCE_CONFIGS.items():
    df_source, df_log = generate_source_txt(config)
    all_source_dfs.append(df_source)
    all_log_dfs.append(df_log)


df_all_sources_review = pd.concat(all_source_dfs, ignore_index=True)
df_all_logs = pd.concat(all_log_dfs, ignore_index=True)

df_all_sources_review.to_csv(SOURCES / "all_sources_review.csv", index=False)
df_all_logs.to_csv(SOURCES / "all_source_page_logs.csv", index=False)

print(f"Saved combined review: {len(df_all_sources_review):,} total source rows.")

### Sanity Check

Verify the combined source review and log files: count of extracted URLs per source and number of links found per source page.

In [ ]:
review_path = SOURCES / "all_sources_review.csv"
logs_path = SOURCES / "all_source_page_logs.csv"

df_review = pd.read_csv(review_path)
df_logs = pd.read_csv(logs_path)

print("Sources found:")
print(df_review["source"].value_counts())
print()
print("Total extracted source rows:", len(df_review))
print()
print("Page log summary:")
print(df_logs.groupby("source")["links_found"].sum())

## 4. Combine Source Lists

Merge all per-source TXT files into a single `symbol_sources.txt` file, deduplicating entries while preserving order.

In [ ]:
source_folder = SOURCES
combined_path = DOCS / "symbol_sources.txt"

if not source_folder.exists():
    raise FileNotFoundError(f"Could not find source folder: {source_folder}")

all_lines = []

for txt_file in sorted(source_folder.glob("*.txt")):
    with open(txt_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            all_lines.append(line)

# Deduplicate (preserving order)
seen = set()
unique_lines = []
for line in all_lines:
    if line not in seen:
        unique_lines.append(line)
        seen.add(line)

os.makedirs(DOCS, exist_ok=True)

with open(combined_path, "w", encoding="utf-8") as f:
    f.write("# Combined symbol source file\n")
    f.write("# Format: website,slug OR full_url\n")
    f.write("# Generated from documents/sources/*.txt\n")
    f.write("\n")
    for line in unique_lines:
        f.write(line + "\n")

print(f"Saved {len(unique_lines):,} unique entries to {combined_path} (from {len(all_lines):,} raw lines).")

## 5. Scrape and Parse Individual Pages

Each URL from `symbol_sources.txt` is scraped with Firecrawl. The raw markdown is parsed into structured records with `name`, `body`, `source`, and `label` fields. Long records are split into overlapping token-bounded chunks (max 64 tokens) before saving.

In [ ]:
# Helper functions for page normalization and parsing

def normalize_firecrawl_page(page, requested_url=None):
    markdown = get_field(page, "markdown", "") or ""
    metadata = get_field(page, "metadata", {}) or {}

    if isinstance(metadata, dict):
        url = (
            metadata.get("sourceURL")
            or metadata.get("source_url")
            or metadata.get("url")
            or requested_url
            or ""
        )
        title = metadata.get("title", "") or ""
        status_code = metadata.get("statusCode") or metadata.get("status_code") or ""
    else:
        url = (
            getattr(metadata, "sourceURL", None)
            or getattr(metadata, "source_url", None)
            or getattr(metadata, "url", None)
            or requested_url
            or ""
        )
        title = getattr(metadata, "title", "") or ""
        status_code = getattr(metadata, "statusCode", None) or getattr(metadata, "status_code", None) or ""

    return {
        "markdown": markdown,
        "metadata": {"url": url, "title": title, "status_code": status_code},
    }


def get_metadata(page):
    meta = page.get("metadata", {}) or {}
    return {
        "url": meta.get("url", "") or "",
        "title": meta.get("title", "") or "",
        "status_code": meta.get("status_code", "") or "",
    }


def clean_markdown_body(body):
    body = str(body)
    body = re.sub(r"!\[.*?\]\(.*?\)", " ", body)
    body = re.sub(r"\[([^\]]+)\]\([^\)]+\)", r"\1", body)
    ui_junk_patterns = [
        r"Open Sharing Options", r"Close Sharing Options", r"Print Content",
        r"Share on Facebook", r"Share on Twitter", r"Share on LinkedIn",
        r"Email this page", r"Copy Link", r"Copied to clipboard",
        r'[\"\u201c\u201d]*\(?opens in a new (?:window|tab)\)?[\"\u201c\u201d]*',
    ]
    for pat in ui_junk_patterns:
        body = re.sub(pat, " ", body, flags=re.IGNORECASE)
    body = re.sub(r"#{1,6}\s+", " ", body)
    body = re.sub(r"\*{1,2}([^*]+)\*{1,2}", r"\1", body)
    body = re.sub(r"`([^`]+)`", r"\1", body)
    body = re.sub(r"\(\s*\)", " ", body)
    body = re.sub(r'"\s*"', " ", body)
    body = re.sub(r"\s+", " ", body).strip()
    return body


# Read website + slug sources from TXT file
source_txt = DOCS / "symbol_sources.txt"

if not os.path.exists(source_txt):
    raise FileNotFoundError(
        f"Could not find {source_txt}.\n\n"
        "Create this file with lines like:\n"
        "https://www.adl.org/resources/hate-symbol,88"
    )

source_rows = []

with open(source_txt, "r", encoding="utf-8") as f:
    for line_num, line in enumerate(f, start=1):
        original_line = line
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        parts = [p.strip() for p in line.split(",", maxsplit=1)]
        if len(parts) != 2:
            continue
        website, slug = parts
        website = website.rstrip("/")
        slug = slug.strip("/")
        if not website or not slug:
            continue
        url = f"{website}/{slug}"
        website_l = website.lower()
        if "adl.org" in website_l:
            source = "adl"
        elif "splcenter.org" in website_l:
            source = "splc"
        elif "hatebase.org" in website_l:
            source = "hatebase"
        else:
            source = "external"
        source_rows.append({
            "line_num": line_num, "website": website, "slug": slug,
            "url": url, "source": source, "label": "hate",
        })

df_sources = pd.DataFrame(source_rows)

if len(df_sources) == 0:
    raise ValueError(f"No valid source rows found in {source_txt}")

df_sources = df_sources.drop_duplicates(subset=["url"]).reset_index(drop=True)
print(f"Loaded {len(df_sources):,} unique URLs from {source_txt}.")


# Scrape individual source pages
raw_pages = []
failed_urls = []

for i, row in df_sources.iterrows():
    url = row["url"]
    source = row.get("source", "external")
    label = row.get("label", "hate")
    slug = row.get("slug", "")
    website = row.get("website", "")

    try:
        page_raw = firecrawl.scrape(url, formats=["markdown"], only_main_content=True)
        page = normalize_firecrawl_page(page_raw, requested_url=url)
        page["txt_source"] = source
        page["txt_label"] = label
        page["txt_slug"] = slug
        page["txt_website"] = website
        page["txt_line_num"] = row.get("line_num", "")
        md = page["markdown"]
        meta = get_metadata(page)
        is_404_like = (
            "Page Not Found" in md[:500]
            or "couldn't find the page" in md[:1000].lower()
            or "404" in meta["title"].lower()
            or "page not found" in meta["title"].lower()
        )
        if is_404_like:
            failed_urls.append((url, "page not found / invalid slug"))
        elif len(md.strip()) >= 50:
            raw_pages.append(page)
        else:
            failed_urls.append((url, "markdown too short"))
    except Exception as e:
        failed_urls.append((url, f"{type(e).__name__}: {e}"))
        print(f"  FAILED {url}: {type(e).__name__}: {e}")

print(f"Scraped {len(raw_pages):,} pages successfully; {len(failed_urls):,} failed.")


# Parse markdown into records
skip_reasons = Counter()

def clean_splc_name(name, slug=""):
    name = str(name).strip()
    name = re.sub(r"\s*\|.*$", "", name).strip()
    name = re.sub(r"\s*- Southern Poverty Law Center.*$", "", name).strip()
    name = re.sub(r"\s*Southern Poverty Law Center\s*", "", name).strip()
    name = re.sub(r"\s+", " ", name).strip()
    bad_names = {"", "in this article", "notifications", "extremist files", "share", "related", "resources"}
    if name.lower() in bad_names:
        name = slug.replace("-", " ").title()
    return name


def extract_splc_record(md, meta_title, url, slug, website, label="hate", debug=False, prefix=""):
    md = re.sub(r"(?is)^.*?Toggle Search\s*", " ", md)
    md = re.sub(r"(?is)^.*?Open Menu\s*", " ", md)
    lines = [line.strip() for line in md.splitlines() if line.strip()]
    if len(lines) == 0:
        skip_reasons["splc_no_lines"] += 1
        return None
    bad_line_starts = (
        "related:", "born:", "location:", "share", "open dialog",
        "in this article", "notifications", "citations", "get the latest",
        "email", "subscribe",
    )
    name = ""
    if meta_title:
        candidate = clean_splc_name(meta_title, slug)
        if candidate and candidate.lower() not in {"extremist files", "in this article"}:
            name = candidate
    if not name:
        for line in lines[:30]:
            line_clean = re.sub(r"^#{1,6}\s*", "", line).strip()
            line_l = line_clean.lower()
            if any(line_l.startswith(x) for x in bad_line_starts):
                continue
            if "splc-extremist-files" in line_l:
                continue
            if len(line_clean) < 2:
                continue
            name = clean_splc_name(line_clean, slug)
            break
    if not name:
        name = slug.replace("-", " ").title()
    name = clean_splc_name(name, slug)
    body = md
    name_pattern = re.escape(name)
    body = re.sub(rf"(?is)^.*?{name_pattern}\s*", " ", body, count=1)
    body = re.sub(r"(?im)^Related:\s*.*$", " ", body)
    body = re.sub(r"(?im)^Born:\s*.*$", " ", body)
    body = re.sub(r"(?im)^Location:\s*.*$", " ", body)
    body = re.sub(r"(?im)^Share\s*$", " ", body)
    body = re.sub(r"(?im)^\(Open dialog with sharing options\)\s*$", " ", body)
    body = re.sub(r"(?im)^SPLC-Extremist-Files-[^\n]+$", " ", body)
    body = re.sub(r"(?im)^In this article\s*$", " ", body)
    body = re.sub(r"(?im)^\s*In his own words\s*$", " In his own words ", body)
    body = re.sub(r"(?im)^\s*Background\s*$", " Background ", body)
    body = re.sub(r"(?is)\bCitations\s+Get the latest updates from.*$", " ", body)
    body = re.sub(r"(?is)\bGet the latest updates from\s+Southern Poverty Law Center.*$", " ", body)
    body = re.sub(r"(?is)\bRacial Justice Issues\s+Find Resources.*$", " ", body)
    body = re.sub(r"(?is)\bPrivacy & Terms\s+Accessibility Statement.*$", " ", body)
    body = re.sub(
        r"(?is)Some areas of this page may shift around if you resize the browser window\.?\s*Be sure to check heading and document order\.?",
        " ", body,
    )
    body = clean_markdown_body(body)
    body = re.sub(r"\bSkip to content\b", " ", body, flags=re.IGNORECASE)
    body = re.sub(r"\bClose Alert\b", " ", body, flags=re.IGNORECASE)
    body = re.sub(r"\bSPLC Home\b", " ", body, flags=re.IGNORECASE)
    body = re.sub(r"\bToggle Search\b", " ", body, flags=re.IGNORECASE)
    body = re.sub(r"\bOpen Menu\b", " ", body, flags=re.IGNORECASE)
    body = re.sub(r"\bIn this article\b", " ", body, flags=re.IGNORECASE)
    body = re.sub(r"\s+", " ", body).strip()
    if len(body) < 50:
        skip_reasons["splc_body_too_short"] += 1
        return None
    return {"name": name, "body": body, "url": url, "source": "splc", "label": label, "slug": slug, "website": website}


def extract_record(page, debug=False, page_idx=None):
    md = page["markdown"] or ""
    meta = get_metadata(page)
    url = meta["url"]
    meta_title = meta["title"]
    source = page.get("txt_source", "external")
    label = page.get("txt_label", "hate")
    slug = page.get("txt_slug", "")
    website = page.get("txt_website", "")
    prefix = f"[PARSE PAGE {page_idx}] " if page_idx is not None else ""
    if len(md.strip()) < 50:
        skip_reasons["markdown_too_short"] += 1
        return None
    if "Page Not Found" in md[:500] or "couldn't find the page" in md[:1000].lower():
        skip_reasons["page_not_found"] += 1
        return None
    if source == "splc":
        rec = extract_splc_record(
            md=md, meta_title=meta_title, url=url, slug=slug,
            website=website, label=label, debug=debug, prefix=prefix,
        )
        if rec is None:
            return None
        skip_reasons["parsed_successfully"] += 1
        return rec
    heading_match = re.search(r"^#{1,3}\s+(.+)", md, re.MULTILINE)
    if heading_match:
        name = heading_match.group(1).strip()
        body = md[heading_match.end():].strip()
    elif meta_title:
        name = str(meta_title).strip()
        body = md.strip()
    else:
        name = slug.replace("-", " ").strip()
        body = md.strip()
    name = re.sub(r"\s*\|.*$", "", name).strip()
    name = re.sub(r"\s*- Anti-Defamation League.*$", "", name).strip()
    name = re.sub(r"\s*Hate Symbols Database\s*", "", name).strip()
    name = re.sub(r"\s+", " ", name).strip()
    if not name:
        skip_reasons["missing_name"] += 1
        return None
    bad_titles = [
        "search", "browse", "all symbols", "hate symbols database",
        "hate symbols", "page not found", "in this article", "notifications",
    ]
    if name.lower().strip() in bad_titles:
        skip_reasons["index_or_listing_page"] += 1
        return None
    body = clean_markdown_body(body)
    if len(body) < 50:
        skip_reasons["body_too_short_after_cleaning"] += 1
        return None
    skip_reasons["parsed_successfully"] += 1
    return {"name": name, "body": body, "url": url, "source": source, "label": label, "slug": slug, "website": website}


parsed = []
for i, page in enumerate(raw_pages):
    rec = extract_record(page, page_idx=i)
    if rec:
        parsed.append(rec)

print(f"Parsed {len(parsed):,} records from {len(raw_pages):,} scraped pages.")
print("Skip/parse reasons:", dict(skip_reasons.most_common()))


# Build initial chunks
records = []
for rec in parsed:
    label = rec.get("label", "hate")
    source = rec.get("source", "external")
    text = f"[{label}] {rec['name']}: {rec['body']}"
    text = re.sub(r"\s+", " ", text).strip()
    records.append({
        "text": text, "source": source, "label": label, "url": rec["url"],
        "symbol_name": rec["name"], "slug": rec.get("slug", ""), "website": rec.get("website", ""),
    })

df_initial = pd.DataFrame(records)
df_initial.insert(0, "chunk_id", range(len(df_initial)))


# Token length check before splitting
tokenizer = AutoTokenizer.from_pretrained("GroNLP/hateBERT")

token_lengths = [len(tokenizer.encode(t, add_special_tokens=False)) for t in df_initial["text"]]
df_initial["token_length"] = token_lengths

plt.figure(figsize=(10, 4))
plt.hist(token_lengths, bins=40)
plt.xlabel("Token length")
plt.ylabel("Count")
plt.title("External chunks — token length distribution before splitting")
plt.tight_layout()
plt.show()


# Split chunks that are too long into smaller windows with overlap
MAX_TOKENS = 50
STRIDE_TOKENS = 25
SAFETY_MARGIN = 8


def token_len(text):
    return len(tokenizer.encode(text, add_special_tokens=False))


def force_token_limit(text, max_tok=50):
    text = re.sub(r"\s+", " ", str(text)).strip()
    while token_len(text) > max_tok:
        toks = tokenizer.encode(text, add_special_tokens=False)
        toks = toks[:max_tok]
        text = tokenizer.decode(toks, skip_special_tokens=True)
        text = re.sub(r"\s+", " ", text).strip()
        if token_len(text) <= max_tok:
            break
        max_tok -= 1
    return text


def split_into_windows(name, body, label="hate", max_tok=50, stride=25):
    prefix = f"[{label}] {name}: "
    prefix_toks = tokenizer.encode(prefix, add_special_tokens=False)
    body_toks = tokenizer.encode(body, add_special_tokens=False)
    budget = max_tok - len(prefix_toks) - SAFETY_MARGIN
    if budget <= 0:
        combined_toks = prefix_toks + body_toks
        text = tokenizer.decode(combined_toks[:max_tok], skip_special_tokens=True)
        return [force_token_limit(text, max_tok)]
    windows = []
    start = 0
    while start < len(body_toks):
        window_toks = body_toks[start:start + budget]
        if not window_toks:
            break
        window_text = tokenizer.decode(window_toks, skip_special_tokens=True)
        text = prefix + window_text
        text = re.sub(r"\s+", " ", text).strip()
        text = force_token_limit(text, max_tok)
        windows.append(text)
        if start + budget >= len(body_toks):
            break
        start += stride
    return windows


final_records = []
for rec in parsed:
    label = rec.get("label", "hate")
    source = rec.get("source", "external")
    full_text = f"[{label}] {rec['name']}: {rec['body']}"
    full_text = re.sub(r"\s+", " ", full_text).strip()
    n_tok = token_len(full_text)
    if n_tok <= MAX_TOKENS:
        final_records.append({
            "text": force_token_limit(full_text, MAX_TOKENS),
            "source": source, "label": label, "url": rec["url"],
            "symbol_name": rec["name"], "slug": rec.get("slug", ""),
            "website": rec.get("website", ""), "original_token_length": n_tok,
        })
    else:
        windows = split_into_windows(
            name=rec["name"], body=rec["body"], label=label,
            max_tok=MAX_TOKENS, stride=STRIDE_TOKENS,
        )
        for window in windows:
            final_records.append({
                "text": force_token_limit(window, MAX_TOKENS),
                "source": source, "label": label, "url": rec["url"],
                "symbol_name": rec["name"], "slug": rec.get("slug", ""),
                "website": rec.get("website", ""), "original_token_length": n_tok,
            })

df_final = pd.DataFrame(final_records)
df_final.insert(0, "chunk_id", range(len(df_final)))
lengths_after = [token_len(t) for t in df_final["text"]]
df_final["token_length"] = lengths_after

assert max(lengths_after) <= MAX_TOKENS, "Some chunks still exceed MAX_TOKENS!"
print(f"Created {len(df_final):,} chunks (from {len(parsed):,} records). Max token length: {max(lengths_after)}.")

plt.figure(figsize=(10, 4))
plt.hist(lengths_after, bins=40)
plt.xlabel("Token length")
plt.ylabel("Count")
plt.title("External chunks — token length distribution after splitting")
plt.tight_layout()
plt.show()


# Save ADL + SPLC page chunks
os.makedirs(DOCS, exist_ok=True)
df_page_chunks = df_final.copy()
df_page_chunks[["chunk_id", "text"]].to_csv(DOCS / "external_symbols_adl_splc.csv", index=False)
df_page_chunks.to_csv(DOCS / "external_symbols_adl_splc_with_urls.csv", index=False)
print(f"Saved {len(df_page_chunks):,} ADL+SPLC chunks to {DOCS / 'external_symbols_adl_splc.csv'}")

## 6. Hatebase Extraction

Hatebase is scraped by applying eight category filters (African, African American, Arabs, homosexuals, female, transgender, Islam, Judaism) and parsing the result tables. Terms are saved to `corpus/knowledge_documents/sources/hatebase_terms.csv`.

In [ ]:
def clean_hatebase_cell(text):
    text = str(text).strip()
    text = re.sub(r"<br\s*/?>", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"\[([^\]]+)\]\([^\)]+\)", r"\1", text)
    text = re.sub(r"\*+", "", text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def parse_hatebase_rows(markdown):
    rows = []
    for raw_line in markdown.splitlines():
        line = raw_line.strip()
        if not line or not line.startswith("|"):
            continue
        if re.match(r"^\|\s*:?-+:?\s*(\|\s*:?-+:?\s*)+\|?$", line):
            continue
        parts = [p.strip() for p in line.strip("|").split("|")]
        if len(parts) < 4:
            continue
        term = clean_hatebase_cell(parts[0])
        language = clean_hatebase_cell(parts[1])
        sightings = clean_hatebase_cell(parts[2])
        offensiveness = clean_hatebase_cell(parts[3])
        if not term or term.lower() in {"term", "vocabulary", "read faqs"}:
            continue
        if language.lower() in {"language", "select"}:
            continue
        if not language and not sightings and not offensiveness:
            continue
        rows.append({"term": term, "language": language, "sightings": sightings, "offensiveness": offensiveness})
    return rows


def extract_showing_range(markdown):
    match = re.search(
        r"Showing\s+([\d,]+)\s*-\s*([\d,]+)\s+of\s+([\d,]+)\s+results",
        markdown, flags=re.IGNORECASE,
    )
    if not match:
        return None
    return {"start": match.group(1), "end": match.group(2), "total": match.group(3), "raw": match.group(0)}


hatebase_filters = [
    {"filter_group": "language", "filter_name": "english_all", "target_category": "english",
     "url": "https://hatebase.org/search_results/language_id%3Deng"},
    {"filter_group": "ethnicity", "filter_name": "african", "target_category": "African",
     "url": "https://hatebase.org/search_results/ethnicity_id%3D300%7Clanguage_id%3Deng"},
    {"filter_group": "ethnicity", "filter_name": "african_american", "target_category": "African American",
     "url": "https://hatebase.org/search_results/ethnicity_id%3D5%7Clanguage_id%3Deng"},
    {"filter_group": "ethnicity", "filter_name": "arabs", "target_category": "Arabs",
     "url": "https://hatebase.org/search_results/ethnicity_id%3D14%7Clanguage_id%3Deng"},
    {"filter_group": "sexual_orientation", "filter_name": "homosexuals", "target_category": "Homosexuals",
     "url": "https://hatebase.org/search_results/orientation_id%3D3%7Clanguage_id%3Deng"},
    {"filter_group": "gender", "filter_name": "female", "target_category": "Female",
     "url": "https://hatebase.org/search_results/gender_id%3D6%7Clanguage_id%3Deng"},
    {"filter_group": "gender", "filter_name": "transgender", "target_category": "Transgender",
     "url": "https://hatebase.org/search_results/gender_id%3D14%7Clanguage_id%3Deng"},
    {"filter_group": "religion", "filter_name": "islam", "target_category": "Islam",
     "url": "https://hatebase.org/search_results/religion_id%3D11%7Clanguage_id%3Deng"},
    {"filter_group": "religion", "filter_name": "judaism", "target_category": "Judaism",
     "url": "https://hatebase.org/search_results/religion_id%3D14%7Clanguage_id%3Deng"},
]


all_rows = []
page_logs = []

for i, spec in enumerate(hatebase_filters, start=1):
    url = spec["url"]
    filter_group = spec["filter_group"]
    filter_name = spec["filter_name"]
    target_category = spec["target_category"]

    try:
        page = firecrawl.scrape(url, formats=["markdown"], only_main_content=True)
        markdown = get_field(page, "markdown", "") or ""
        rows = parse_hatebase_rows(markdown)
        showing_range = extract_showing_range(markdown)
        showing_raw = showing_range["raw"] if showing_range else ""
        for row in rows:
            row["source"] = "hatebase"
            row["label"] = "hate"
            row["source_url"] = url
            row["filter_group"] = filter_group
            row["filter_name"] = filter_name
            row["target_category"] = target_category
            row["showing_range"] = showing_raw
            row["coverage_note"] = "selected_english_filters"
        all_rows.extend(rows)
        page_logs.append({
            "filter_group": filter_group, "filter_name": filter_name,
            "target_category": target_category, "url": url,
            "markdown_length": len(markdown), "rows_found": len(rows),
            "showing_range": showing_raw, "status": "ok",
        })
    except Exception as e:
        print(f"  FAILED {filter_name}: {type(e).__name__}: {e}")
        page_logs.append({
            "filter_group": filter_group, "filter_name": filter_name,
            "target_category": target_category, "url": url,
            "markdown_length": 0, "rows_found": 0, "showing_range": "",
            "status": f"failed: {type(e).__name__}: {e}",
        })


df_hatebase = pd.DataFrame(all_rows)

if len(df_hatebase) == 0:
    print("No Hatebase rows parsed. Check filter URLs.")
else:
    df_hatebase = df_hatebase.drop_duplicates(subset=["term", "language"]).reset_index(drop=True)
    df_hatebase["text"] = df_hatebase.apply(
        lambda r: (
            f"[{r['label']}] {r['term']}: "
            f"Hatebase entry. Target category: {r['target_category']}. "
            f"Filter group: {r['filter_group']}. "
            f"Language: {r['language']}. "
            f"Offensiveness: {r['offensiveness']}. "
            f"Sightings: {r['sightings']}."
        ),
        axis=1,
    )
    os.makedirs(SOURCES, exist_ok=True)
    csv_path = SOURCES / "hatebase_terms.csv"
    log_path = SOURCES / "hatebase_filter_logs.csv"
    df_hatebase.to_csv(csv_path, index=False)
    pd.DataFrame(page_logs).to_csv(log_path, index=False)
    print(f"Saved {len(df_hatebase):,} Hatebase terms to {csv_path}")

### Diagnostics

Basic checks on the extracted Hatebase terms: deduplication stats, counts by filter group and target category, and detection of potential issues (non-English rows, HTML artifacts).

In [ ]:
if "df_hatebase" in globals() and isinstance(df_hatebase, pd.DataFrame):
    hb = df_hatebase.copy()
elif os.path.exists(SOURCES / "hatebase_terms.csv"):
    hb = pd.read_csv(SOURCES / "hatebase_terms.csv")
else:
    raise FileNotFoundError("No df_hatebase found and no hatebase_terms.csv file exists.")

print("Total rows:", len(hb))

if {"term", "language"}.issubset(hb.columns):
    unique_term_lang = hb.drop_duplicates(subset=["term", "language"])
    print("Unique term-language pairs:", len(unique_term_lang))
    print("Duplicate term-language rows:", len(hb) - len(unique_term_lang))

if "language" in hb.columns:
    non_english = hb[hb["language"].fillna("").str.lower() != "english"]
    print("Non-English rows:", len(non_english))

if "term" in hb.columns:
    html_terms = hb[hb["term"].astype(str).str.contains(r"<[^>]+>", regex=True, na=False)]
    print("Terms still containing HTML tags:", len(html_terms))

if "text" in hb.columns:
    empty_text = hb[hb["text"].astype(str).str.strip() == ""]
    print("Empty text rows:", len(empty_text))

## 7. Final Combine and Save

Merge ADL, SPLC, and Hatebase chunks into a single dataset. Three files are saved: the pipeline-ready chunk file `chunks_knowledge.csv` (chunk_id | text), the full provenance file with URLs, and a clean review CSV.

In [ ]:
# Retrieve ADL + SPLC chunks
df_page_chunks = df_final.copy()

# Retrieve and convert Hatebase rows to the same chunk structure as ADL/SPLC
df_hb_source = None

if (
    "df_hatebase" in globals()
    and isinstance(df_hatebase, pd.DataFrame)
    and len(df_hatebase) > 0
):
    df_hb_source = df_hatebase.copy()
elif os.path.exists(SOURCES / "hatebase_terms.csv"):
    df_hb_source = pd.read_csv(SOURCES / "hatebase_terms.csv")
else:
    print("No Hatebase rows found. Final file will include ADL + SPLC only.")


chunks_to_combine = [df_page_chunks]

if df_hb_source is not None and len(df_hb_source) > 0:
    df_hb_chunks = df_hb_source.copy()
    df_hb_chunks["source"] = "hatebase"
    if "label" not in df_hb_chunks.columns:
        df_hb_chunks["label"] = "hate"
    if "coverage_note" not in df_hb_chunks.columns:
        df_hb_chunks["coverage_note"] = "selected_english_filters"
    if "term" in df_hb_chunks.columns:
        df_hb_chunks["symbol_name"] = df_hb_chunks["term"].astype(str)
    elif "symbol_name" not in df_hb_chunks.columns:
        df_hb_chunks["symbol_name"] = ""
    if "source_url" in df_hb_chunks.columns:
        df_hb_chunks["url"] = df_hb_chunks["source_url"]
    elif "url" not in df_hb_chunks.columns:
        df_hb_chunks["url"] = "https://hatebase.org/search_results/"
    df_hb_chunks["website"] = "https://hatebase.org/search_results"
    if "slug" not in df_hb_chunks.columns:
        df_hb_chunks["slug"] = (
            df_hb_chunks["symbol_name"].astype(str)
            .str.lower().str.replace(r"\s+", "-", regex=True)
            .str.replace(r"[^a-z0-9\-]+", "", regex=True).str.strip("-")
        )
    for col in ["language", "offensiveness", "sightings", "filter_group",
                "filter_name", "target_category", "showing_range"]:
        if col not in df_hb_chunks.columns:
            df_hb_chunks[col] = ""
    df_hb_chunks["body"] = df_hb_chunks.apply(
        lambda r: (
            f"Hatebase entry for {r['symbol_name']}. "
            f"Target category: {r.get('target_category', '')}. "
            f"Filter group: {r.get('filter_group', '')}. "
            f"Language: {r.get('language', '')}. "
            f"Offensiveness: {r.get('offensiveness', '')}. "
            f"Sightings: {r.get('sightings', '')}."
        ),
        axis=1,
    )
    df_hb_chunks["text"] = df_hb_chunks.apply(
        lambda r: re.sub(r"\s+", " ", f"[{r['label']}] {r['symbol_name']}: {r['body']}").strip(),
        axis=1,
    )
    if "MAX_TOKENS" in globals() and "force_token_limit" in globals():
        df_hb_chunks["text"] = df_hb_chunks["text"].apply(lambda t: force_token_limit(t, MAX_TOKENS))
    if "token_len" in globals():
        df_hb_chunks["token_length"] = [token_len(t) for t in df_hb_chunks["text"]]
    else:
        df_hb_chunks["token_length"] = ""
    df_hb_chunks["original_token_length"] = df_hb_chunks["token_length"]
    for col in df_page_chunks.columns:
        if col not in df_hb_chunks.columns:
            df_hb_chunks[col] = ""
    for col in df_hb_chunks.columns:
        if col not in df_page_chunks.columns:
            df_page_chunks[col] = ""
    df_hb_chunks = df_hb_chunks[df_page_chunks.columns]
    chunks_to_combine = [df_page_chunks, df_hb_chunks]


df_external_final = pd.concat(chunks_to_combine, ignore_index=True)
df_external_final["chunk_id"] = range(len(df_external_final))

df_final_pipeline = df_external_final[["chunk_id", "text"]].copy()


# Save final outputs
os.makedirs(DOCS, exist_ok=True)

df_final_pipeline.to_csv(DOCS / "final_external_symbols.csv", index=False)
df_external_final.to_csv(DOCS / "final_external_symbols_with_urls.csv", index=False)

review_columns = [
    "chunk_id", "symbol_name", "label", "source", "coverage_note", "text",
    "token_length", "original_token_length", "website", "slug", "url",
    "language", "offensiveness", "sightings", "filter_group", "filter_name",
    "target_category", "showing_range",
]
review_columns = [col for col in review_columns if col in df_external_final.columns]
df_external_final[review_columns].to_csv(DOCS / "final_external_symbols_clean_review.csv", index=False)

CHUNKS.mkdir(parents=True, exist_ok=True)
df_final_pipeline.to_csv(CHUNKS / "chunks_knowledge.csv", index=False)
print(f"Saved {len(df_final_pipeline):,} chunks to {CHUNKS / 'chunks_knowledge.csv'}")